In [ ]:
# -----------------------------------------------
# 📦 Imports
# -----------------------------------------------
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import current_timestamp
import uuid
import datetime

In [ ]:
# -----------------------------------------------
# 🧱 Step 1: Define Schemas and Table Names
# -----------------------------------------------
Lakehouse_name ='LH_RAW_E2E_DA_Weather_Data'
raw_schema = "raw" #Bronge
log_schema = "log" 
refined_schema="refined" #Silver
curated_schema = "curated" #Gold
log_table_name = "monitoring_log"

In [ ]:
qualified_log_table = f"{log_schema}.{log_table_name}"

In [ ]:
# -----------------------------------------------
# 🧾 Step 2: Define Log Table Structure
# -----------------------------------------------
log_table_schema = StructType([
    StructField("notebook_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("operation_type", StringType(), True),
    StructField("row_count", IntegerType(), True),
    StructField("message", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

In [ ]:
# -----------------------------------------------
# 🧪 Step 3: Utility – Check and Create Schema
# -----------------------------------------------
def ensure_schema_exists(schema_name: str, lakehouse_name: str):
    print(f"🔍 Checking if schema '{schema_name}' exists in Lakehouse '{lakehouse_name}'...")

    # Get all schemas and filter by lakehouse name
    schemas_raw = [row["namespace"] for row in spark.sql("SHOW SCHEMAS").collect()]
    matching_schemas = [
        ns for ns in schemas_raw if ns.lower() == f"{lakehouse_name}.{schema_name}".lower()
    ]

    if not matching_schemas:
        print(f"📁 Schema '{schema_name}' not found. Creating it...")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{schema_name}")
        print(f"✅ Created schema: {lakehouse_name}.{schema_name}")
    else:
        print(f"✅ Schema '{lakehouse_name}.{schema_name}' already exists.")

In [ ]:
# -----------------------------------------------
# 🧱 Step 4: Ensure Required Schemas Exist
# -----------------------------------------------
ensure_schema_exists("raw", Lakehouse_name) 
ensure_schema_exists("refined", Lakehouse_name)
ensure_schema_exists("curated", Lakehouse_name)
ensure_schema_exists("log", Lakehouse_name)

In [ ]:
# -----------------------------------------------
# 📋 Step 5: Check and Create Log Table
# -----------------------------------------------
print(f"🔍 Checking if table '{qualified_log_table}' exists...")
table_rows = spark.sql(f"SHOW TABLES IN {log_schema}").collect()
existing_tables = [row["tableName"] for row in table_rows]

if log_table_name not in existing_tables:
    print(f"📋 Table '{qualified_log_table}' does not exist. Creating...")
    empty_df = spark.createDataFrame([], log_table_schema)
    empty_df.write.format("delta").mode("overwrite").saveAsTable(qualified_log_table)
    print(f"✅ Created log table: {qualified_log_table}")
else:
    print(f"✅ Log table '{qualified_log_table}' already exists.")

In [ ]:
# -----------------------------------------------
# ✍️ Step 6: Insert Sample Log Row
# -----------------------------------------------
print("📝 Inserting a sample log row...")

sample_row = [{
    "notebook_name": "NB_00_Setup_Lakehouse",
    "table_name": qualified_log_table,
    "run_id": str(uuid.uuid4()),
    "user_name": "initialiser",
    "operation_type": "initial_setup",
    "row_count": 0,
    "message": "Log table initialised",
    "error_message": "",
    "timestamp": datetime.datetime.utcnow()
}]
sample_df = spark.createDataFrame(sample_row, schema=log_table_schema)

sample_df.write.format("delta").mode("append").saveAsTable(qualified_log_table)

print("✅ Sample row inserted successfully.")

In [ ]:
# -----------------------------------------------
# 📊 Step 7: Preview Log Table
# -----------------------------------------------
print("🔍 Previewing log table:")
df = spark.sql(f"SELECT * FROM {qualified_log_table} ORDER BY timestamp DESC LIMIT 100")
display(df)